# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [1]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

HypEx 1.0.7 · pandas 2.2.3


### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [2]:
VARIANT = None   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [3]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

вариант 25: «Кэшбэк 5 % на аптеку»
6000 клиентов · test 3030 · control 2970


### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [4]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

проверка формата готова


## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [5]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

клиентов на группу: 3000, σ оборота: 3276 ₽, ρ двух прошлых периодов: 0.592
MDE без снижения дисперсии: 168 ₽
MDE с CUPED:                135 ₽
+3 % к среднему обороту:    96 ₽


#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Если добавить кешбек 5% на аптеки, для владельцев подписки, то оборот по подписке изменится не меньше чем на 168 рублей за 4 недели, потому что повысится мотивация тратить больше в аптеках благодаря кешбэку. |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Владелец подписки, так как кешбэк подключается на уровне аккаунта и не расшеривается между разными пользователями |
| группы и доли | 2 группы: control - без кешбэка 5% на аптеки и treatment - кешбэк 5% на аптеки, по 3000 клиентов на группу |
| основная метрика | Средний оборот по подписке на пользователя за период эксперимента в рублях |
| метрики-стражи: что не должно ухудшиться | Средний бюджет на кешбэк, чтобы промо не ушло в убыток, доля отмен и возвратов в аптеках, конверсия в продление базовой подписки и отток. |
| α, мощность, MDE — числами из ячейки выше | α = 0.05, мощность (1 − β) = 0.80, MDE без снижения дисперсии = 168 ₽, MDE с CUPED = 135 ₽, ориентир +3% к среднему обороту или 96 ₽. |
| поправка на множественные сравнения | Не требуется, так как у нас одна основная метрика и одна пары групп (treatment vs control), для метрик-стражей контролируем отдельно пороги безопасности. |
| оценщик | CUPED-оценщик разности средних, t-тест на ковариатно-скорректированной метрике оборота с ковариатой из предэкспериментального периода, где корреляция ρ = 0.592. |
| окно | окно эксперимента: 28 дней с момента активации кешбэка, ковариата - аналогичный 28-дневный интервал до старта акции. |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | Запускаем если статистически значимый рост основной метрики (p < 0.05) с эффектом не ниже MDE и без ухудшения метрик-стражей. Дорабатываем если положительный, но недостаточный или пограничный эффект (например, без статзначимости при сохранении стражей), тогда корректируем размер кешбэка, таргетинг или длительность. Отказываемся если отсутствует значимый положительный эффект или есть значимое ухудшение метрик-стражей или экономики акции. |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [6]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

{'control': 2970, 'test': 3030}
scipy:    p = 0.4386
pilot.py: p = 0.4386


0.439

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [7]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

spend_prev: test 3253.49 ₽, control 3121.30 ₽
p = 0.1178


0.118

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [8]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

test 3680.63 ₽, control 3231.67 ₽
разность 448.95 ₽, интервал 271.49 … 626.42, p = 0.0000


[448.95, 271.49, 626.42]

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [9]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

feature group  control mean  test mean  difference  difference % TTest pass  TTest p-value UTest pass  UTest p-value
  spend  test    3231.67271 3680.62767   448.95496      13.89234         OK   7.307451e-07         OK        0.00001


[0.0, 0.0]

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [10]:
# Числа для ответа на вопрос 1 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

p_srm = float(srm.pvalue)
p_balance = float(balance.pvalue)

effect_diff = float(effect["diff"])
effect_lo = float(effect["lo"])
effect_hi = float(effect["hi"])

p_ttest = float(ab["TTest p-value"].iloc[0])
p_utest = float(ab["UTest p-value"].iloc[0])

print(f"SRM p-value: {p_srm:.4f}")
print(f"Balance p-value: {p_balance:.4f}")
print(f"Effect: {effect_diff:.2f} ₽")
print(f"95% CI: [{effect_lo:.2f}; {effect_hi:.2f}]")
print(f"MDE: {mde_raw:.2f} ₽")
print(f"MDE with CUPED: {mde_cuped:.2f} ₽")
print(f"t-test p-value: {p_ttest:.8f}")
print(f"U-test p-value: {p_utest:.8f}")


SRM p-value: 0.4386
Balance p-value: 0.1178
Effect: 448.95 ₽
95% CI: [271.49; 626.42]
MDE: 167.56 ₽
MDE with CUPED: 135.03 ₽
t-test p-value: 0.00000073
U-test p-value: 0.00000990


#### ✍️ Ответ на вопрос 1

Разбиению на группы можно доверять, так как SRM-тест не выявил значимого отклонения от пропорции 50/50 (p = 0.4386), а различия в обороте до эксперимента также статистически незначимы (p = 0.1178). Эффект составил 448.95 рублей, что превышает MDE без CUPED (168 ₽), 95%-й доверительный интервал [271.49; 626.42] не включает ноль, причём его нижняя граница также выше заданного MDE (даже без CUPED). Оба критерия в задаче 4 подтвердили статистическую значимость эффекта: t-test (p = 7.31 × 10⁻⁷) и U-test (p = 0.00001). Различие в величине p-value в двух тестах объясняется тем, что t-test сравнивает средние, а U-test - распределения с использованием рангов, поэтому они по-разному реагируют на асимметрию и выбросы.

## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [11]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table)[1]

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

         control    test    diff      lo      hi       p
metric                                                  
churned   0.0283  0.0337  0.0054 -0.0034  0.0141  0.2289
tickets   0.0667  0.0762  0.0096 -0.0037  0.0229  0.1580
churned     0    1
grp               
control  2886   84
test     2928  102
χ² по оттоку: p = 0.2594


[0.54, 0.259, 0.01, 0.158]

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [12]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

HypEx о поправке: There was less than three groups or multitest method wasn't provided
  feature  control mean  test mean  difference  TTest p-value   holm  rejected
    spend     3231.6727  3680.6277    448.9550         0.0000 0.0000      True
purchases        4.8687     5.4776      0.6089         0.0000 0.0000      True
  churned        0.0283     0.0337      0.0054         0.2289 0.3161     False
  tickets        0.0667     0.0762      0.0096         0.1580 0.3161     False


['purchases', 'spend']

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [13]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

theta = 0.662, дисперсия ниже на 38.0%
разность 361.43 ₽, интервал 221.56 … 501.31


[0.662, 38.0, 361.43, 221.56, 501.31]

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [14]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

target covariate variance_reduction
 spend spend_lag              38.0%
    feature group  control mean   test mean  difference  difference % TTest pass  TTest p-value
      spend  test   3231.672710 3680.627670  448.954960     13.892340         OK   7.307451e-07
spend_cuped  test   3275.870718 3637.304871  361.434153     11.033224         OK   4.214610e-07


361.43

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [26]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

ci_width_raw = float(effect["hi"] - effect["lo"])
ci_width_cuped = float(effect_cuped["hi"] - effect_cuped["lo"])
ci_narrowing_pct = (1 - ci_width_cuped / ci_width_raw) * 100

prev_test = float(test["spend_prev"].mean())
prev_control = float(control["spend_prev"].mean())
prev_diff = prev_test - prev_control

print(f"Ширина ДИ до CUPED: {ci_width_raw:.2f} ₽, после CUPED: {ci_width_cuped:.2f} ₽ -> сузился на {ci_narrowing_pct:.1f}%")
print(f"Эффект до CUPED: {effect['diff']:.2f} ₽, после CUPED: {effect_cuped['diff']:.2f} ₽ -> разница: {effect['diff'] - effect_cuped['diff']:.2f} рублей")
print(f"Средний spend_prev: test = {prev_test:.2f} ₽, control = {prev_control:.2f} ₽ -> разница: {prev_diff:.2f} рублей")

Ширина ДИ до CUPED: 354.93 ₽, после CUPED: 279.74 ₽ -> сузился на 21.2%
Эффект до CUPED: 448.95 ₽, после CUPED: 361.43 ₽ -> разница: 87.52 рублей
Средний spend_prev: test = 3253.49 ₽, control = 3121.30 ₽ -> разница: 132.19 рублей


#### ✍️ Ответ на вопрос 2

Ни одна из метрик-стражей не сработала: изменения оттока и обращений в поддержку незначимы, а после поправки Холма значимыми остались только оборот и покупки. Применение CUPED сузило 95%-й доверительный интервал на 21.2%, при этом точечная оценка эффекта снизилась с 448.95 рублей до 361.43 рублей. Это произошло из-за случайного перекоса до эксперимента: в предпериоде тестовая группа в среднем тратила на 132.19 ₽ больше контрольной, и CUPED очистил оценку от этого изначального преимущества ($\theta = 0.662$). 

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [16]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.

rng = np.random.default_rng(VARIANT)

x_test = test["spend"].to_numpy()
x_control = control["spend"].to_numpy()

n_bootstrap = 2000
differences = []

for _ in range(n_bootstrap):
    sample_test = rng.choice(x_test, size=len(x_test), replace=True)
    sample_control = rng.choice(x_control, size=len(x_control), replace=True)

    differences.append(sample_test.mean() - sample_control.mean())

ci_bootstrap = np.percentile(differences, [2.5, 97.5])

answer_9 = [round(float(x), 2) for x in ci_bootstrap]
answer_9

[271.43, 634.6]

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [17]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.

candidates = {
    "spend_prev": mine["spend_prev"],
    "spend_prev2": mine["spend_prev2"],
    "spend_prev+spend_prev2": mine["spend_prev"] + mine["spend_prev2"],
    "purchases_prev": mine["purchases_prev"],
    "sessions_prev": mine["sessions_prev"],
    "sessions_prev2": mine["sessions_prev2"],
}


rho_squared = {
    name: mine["spend"].corr(feature) ** 2 * 100
    for name, feature in candidates.items()
}

rho_purchases = mine["spend"].corr(mine["purchases"]) ** 2 * 100

for name, value in sorted(
    rho_squared.items(), key=lambda x: x[1], reverse=True
):
    print(f"{name}: {value:.1f}%")

print(f"\npurchases (после начала эксперимента): {rho_purchases:.1f}%")

best = max(rho_squared, key=rho_squared.get)

answer_10 = [best, round(float(rho_squared[best]), 1)]
answer_10


spend_prev+spend_prev2: 44.5%
sessions_prev: 41.4%
sessions_prev2: 38.8%
spend_prev: 38.0%
spend_prev2: 32.9%
purchases_prev: 29.9%

purchases (после начала эксперимента): 72.1%


['spend_prev+spend_prev2', 44.5]

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [18]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.

mine["treated"] = (mine["grp"] == "test").astype(int)
feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]
roles_r = {"client_id": InfoRole(str), "treated": TreatmentRole(), "spend": TargetRole(),
           **{f: FeatureRole() for f in feats}}

answer_11 = F(Matching(n_neighbors=5).execute(Dataset(roles=dict(roles_r), data=mine, default_role=InfoRole())).resume).loc['ATE']
answer_11 = [round(float(answer_11["Effect Size"]), 2), round(float(answer_11["CI Lower"]), 2), round(float(answer_11["CI Upper"]), 2)]
answer_11

[373.66, 229.23, 518.09]

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [19]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽

c, t = mine[mine["grp"] == "control"], mine[mine["grp"] == "test"]
lift = diff_ci(t["spend"], c["spend"])
ticket_gap = t["tickets"].mean() - c["tickets"].mean()
churn_gap = t["churned"].mean() - c["churned"].mean()

money = pd.Series({
    "комиссия с прироста оборота": answer_7[2] * COMMISSION,
    "дополнительные обращения": -ticket_gap * TICKET_COST,
    "дополнительный отток": -churn_gap * CHURN_COST,
})

money["итого на клиента за 4 недели"] = money.iloc[:3].sum()

answer_12 = round(float(money.iloc[3]), 2)
answer_12

-0.05

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [20]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

# Сравнение ширины 95%-х доверительных интервалов

intervals = {
    "Задача 3 - обычная оценка": answer_3,
    "Задача 7 - CUPED": answer_7[2:],
    "Задача 9 - Bootstrap": [None, *answer_9],
    "Задача 11 - Matching": answer_11,
}

for name, values in intervals.items():
    width = values[2] - values[1]
    print(f"{name}: ширина интервала = {width:.2f} ₽")

print("\nЭкономика акции:")
print(money.round(2))

Задача 3 - обычная оценка: ширина интервала = 354.93 ₽
Задача 7 - CUPED: ширина интервала = 279.75 ₽
Задача 9 - Bootstrap: ширина интервала = 363.17 ₽
Задача 11 - Matching: ширина интервала = 288.86 ₽

Экономика акции:
комиссия с прироста оборота     5.42
дополнительные обращения       -1.44
дополнительный отток           -4.04
итого на клиента за 4 недели   -0.05
dtype: float64


#### ✍️ Ответ на вопрос 3

Больше всего, думаю, стоит, доверять оценке CUPED, так как она получена в случайном эксперименте с поправкой на предпериодный оборот и имеет самый узкий 95%-й доверительный интервал. `purchases` нельзя использовать как ковариату, поскольку это показатель, который мы измерили уже после начала эксперимента: он сам может измениться под влиянием акции, поэтому его использование приведёт к утечке информации и смещению оценки. При заданных допущениях акция не выгодна: расчётный результат составляет −0.05 ₽ на клиента за четыре недели; при этом расчёт не учитывает фактическую стоимость кешбэка, маржинальность операций и другие возможные доходы и расходы.

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту

**Что проверяли:** Провели A/B-тест акции «Кешбэк 5% на аптеки» на 6 000 подписчиках ПРАЙМ в течение четырёх недель. Проверили корректность распределения по группам, сравнили оборот в тесте и контроле, оценили влияние акции на отток и обращения в поддержку.

**Что получилось:** Тестовая группа показала прирост среднего оборота на 448.95 ₽ на клиента. После корректировки CUPED оценка составила 361.43 ₽, 95%-й доверительный интервал — [221.56; 501.31] ₽. Эффект статистически значим. При этом статистически значимого ухудшения метрик-стражей не обнаружено. После поправки Холма значимыми остались оборот и количество покупок.

**Какие остались вопросы:** По заданным допущениям расчётный финансовый результат составляет −0.05 ₽ на клиента за четыре недели. Таким образом, прирост оборота практически полностью компенсируется расходами на дополнительные обращения и отток. При этом расчёт не учитывает фактическую стоимость кешбэка, маржинальность операций, возвраты и влияние акции на продление подписки, поэтому для окончательной оценки экономики нужны дополнительные данные.

**Что рекомендуем:** Не масштабировать акцию на всех подписчиков в текущем виде. Сначала уточнить полную экономику кешбэка, оценить его стоимость и маржинальность аптечных операций, а затем провести дополнительный пилот с изменёнными условиями предложения. Несмотря на статистически значимый рост оборота, текущие расчёты не подтверждают экономическую целесообразность масштабирования.

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [21]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

задача  1: 0.439
    ✅ формат в порядке
задача  2: 0.118
    ✅ формат в порядке
задача  3: [448.95, 271.49, 626.42]
    ✅ формат в порядке
задача  4: [0.0, 0.0]
    ✅ формат в порядке
задача  5: [0.54, 0.259, 0.01, 0.158]
    ✅ формат в порядке
задача  6: ['purchases', 'spend']
    ✅ формат в порядке
задача  7: [0.662, 38.0, 361.43, 221.56, 501.31]
    ✅ формат в порядке
задача  8: 361.43
    ✅ формат в порядке
задача  9: [271.43, 634.6]
    ✅ формат в порядке
задача 10: ['spend_prev+spend_prev2', 44.5]
    ✅ формат в порядке
задача 11: [373.66, 229.23, 518.09]
    ✅ формат в порядке
задача 12: -0.05
    ✅ формат в порядке

Не забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.

--- служебная строка, не удаляйте ---
HW3_ANSWERS={"variant": 25, "answers": {"1": 0.439, "2": 0.118, "3": [448.95, 271.49, 626.42], "4": [0.0, 0.0], "5": [0.54, 0.259, 0.01, 0.158], "6": ["purchases", "spend"], "7": [0.662, 38.0, 361.43, 221.56, 501.31], "8": 361.43, "9": [271.43, 634.6], 

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [22]:
# Исследование А: ваш код. Всё — от mine.

#### ✍️ Исследование А

_Замените эту строку своим текстом, заголовок выше не трогайте. Схема: вопрос → как проверяли → что получилось → что это значит и чего расчёт не доказывает._

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [23]:
# Исследование Б: ваш код. Всё — от mine.

#### ✍️ Исследование Б

_Замените эту строку своим текстом, заголовок выше не трогайте. Схема: вопрос → как проверяли → что получилось → что это значит и чего расчёт не доказывает._